SCRIPT 1: calc_ls_mask.ipynb
- Reads the sftlf (land_area_fraction) from AUS-11 (BARRA-R2) in project ob53.
- Masks land fraction <= 0.75 and saves as NetCDF.

**Reads:** "/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/fx/sftlf/latest/sftlf_AUS-11_ERA5_historical_hres_BOM_BARRA-R2_v1.nc"\
**Writes:** "ID_HW_BARRA/data/preprocess/land_sea_mask.nc"\
**Compute:** Medium (4CPU, 18GB) should be fine as the file is small.\
**Environment:** analysis3

In [1]:
import xarray as xr, netCDF4 as nc, numpy as np, pandas as pd, os
from pathlib import Path

import dask
import dask.array as da
from dask.distributed import LocalCluster, Client
from datetime import datetime

In [2]:
os.chdir('/g/data/ng72/ms5578/ID_HW_BARRA')
workingDir = Path().absolute()
print(f"{workingDir}")

/g/data/ng72/ms5578/ID_HW_BARRA


In [3]:
fpath = "/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/fx/sftlf/latest/sftlf_AUS-11_ERA5_historical_hres_BOM_BARRA-R2_v1.nc"
write_path = f'{workingDir}/data/preprocess/'

In [4]:
ls_file = xr.open_dataset(fpath)
ls_ratio = ls_file.sftlf

lat = ls_ratio['lat'].values  # or ds['latitude']
lon = ls_ratio['lon'].values  # or ds['longitude']

ls_ratio

<xarray.DataArray 'sftlf' (lat: 646, lon: 1082)> Size: 3MB
[698972 values with dtype=float32]
Coordinates:
  * lat      (lat) float64 5kB -57.97 -57.86 -57.75 -57.64 ... 12.76 12.87 12.98
  * lon      (lon) float64 9kB 88.48 88.59 88.7 88.81 ... 207.2 207.3 207.4
    crs      int32 4B ...
Attributes:
    long_name:      Percentage of the grid  cell occupied by land (including ...
    standard_name:  land_area_fraction
    units:          %
    frequency:      fx
    grid_mapping:   crs

In [5]:
ls_mask = ls_ratio.where(ls_ratio > 0.75, 0)
ls_mask = ls_mask.where(ls_ratio <= 0.75, 1).astype(int)

In [6]:
lat_min, lat_max = -45, -10
lon_min, lon_max = 110, 155

In [7]:
# These are assumed names — change if needed
lat = ls_ratio['lat']                # 1D or 2D
lon = ls_ratio['lon']                # 1D or 2D


# Create coordinate-based masks using broadcasting
lat_mask = (ls_ratio['lat'] >= lat_min) & (ls_ratio['lat'] <= lat_max)
lon_mask = (ls_ratio['lon'] >= lon_min) & (ls_ratio['lon'] <= lon_max)

# Combine them using broadcasting (xarray will align dimensions)
# This assumes lat and lon are 2D or broadcastable with the mask
aus_region_mask = lat_mask & lon_mask

# Apply mask: keep land (=1) AND within bounding box, else 0
aus_land_mask = ls_mask.where(
    (ls_mask == 1) & aus_region_mask,
    other=0
)

In [8]:
australia_land_mask

NameError: name 'australia_land_mask' is not defined

In [ ]:
ls_file = aus_land_mask.to_dataset()

In [ ]:
encoding = {v: {"zlib": True, "complevel": 4, "shuffle": True} for v in ls_file.data_vars}

ls_file.to_netcdf(f'{write_path}land_sea_mask.nc',
                   encoding=encoding,
                   engine="netcdf4")